In [2]:
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageDraw, ImageFont, ImageTk
from fontTools.ttLib import TTFont
import math

CELL_SIZE = 48
FONT_SIZE = 28
COLS = 12

In [ ]:
class FontGridApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Font Unicode Picker")

        self.canvas = tk.Canvas(root, width=COLS * CELL_SIZE, height=600)
        self.canvas.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

        self.scrollbar = tk.Scrollbar(root, command=self.canvas.yview)
        self.scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        self.canvas.config(yscrollcommand=self.scrollbar.set)

        self.frame = tk.Frame(self.canvas)
        self.canvas.create_window((0, 0), window=self.frame, anchor="nw")

        self.frame.bind("<Configure>", self.on_frame_configure)

        self.font_path = None
        self.glyphs = []
        self.selected = set()
        self.images = []

        btn_frame = tk.Frame(root)
        btn_frame.pack(fill=tk.X)

        tk.Button(btn_frame, text="Open Font", command=self.load_font).pack(side=tk.LEFT)
        tk.Button(btn_frame, text="Export", command=self.export).pack(side=tk.LEFT)

    def on_frame_configure(self, event):
        self.canvas.configure(scrollregion=self.canvas.bbox("all"))

    def load_font(self):
        path = filedialog.askopenfilename(filetypes=[("Font", "*.ttf *.otf *.woff *.woff2")])
        if not path:
            return
        self.font_path = path

        tt = TTFont(path)
        cmap = tt.getBestCmap()
        self.glyphs = sorted(cmap.keys())

        self.draw_grid()

    def draw_grid(self):
        for w in self.frame.winfo_children():
            w.destroy()

        self.images.clear()
        self.selected.clear()

        try:
            pil_font = ImageFont.truetype(self.font_path, FONT_SIZE)
        except:
            messagebox.showerror("Error", "Font load failed")
            return

        rows = math.ceil(len(self.glyphs) / COLS)

        for idx, code in enumerate(self.glyphs):
            r = idx // COLS
            c = idx % COLS

            img = Image.new("RGB", (CELL_SIZE, CELL_SIZE), "white")
            draw = ImageDraw.Draw(img)

            ch = chr(code)
            bbox = pil_font.getbbox(ch)
            w = bbox[2] - bbox[0]
            h = bbox[3] - bbox[1]
            draw.text(((CELL_SIZE - w) / 2, (CELL_SIZE - h) / 2),
                      ch, fill="black", font=pil_font)

            tk_img = ImageTk.PhotoImage(img)
            self.images.append(tk_img)

            lbl = tk.Label(self.frame, image=tk_img, bd=1, relief="solid")
            lbl.grid(row=r, column=c)
            lbl.bind("<Button-1>", lambda e, cd=code, lb=lbl: self.toggle(cd, lb))

    def toggle(self, code, label):
        if code in self.selected:
            self.selected.remove(code)
            label.config(
                bg="white",
                relief="solid",
                bd=1,
                highlightthickness=0
            )
        else:
            self.selected.add(code)
            label.config(
                bg="#88cfff",        # stronger blue
                relief="raised",
                bd=3,
                highlightbackground="blue",
                highlightthickness=2
            )

    def export(self):
        if not self.selected:
            messagebox.showinfo("Info", "Nothing selected")
            return

        arr = sorted(self.selected)
        text = ", ".join([f"0x{c:X}" for c in arr])

        path = filedialog.asksaveasfilename(defaultextension=".txt")
        if not path:
            return

        with open(path, "w") as f:
            f.write(text)

        messagebox.showinfo("Done", "Exported!")


if __name__ == "__main__":
    root = tk.Tk()
    app = FontGridApp(root)
    root.mainloop()